# **Project PartB: Emotion Recognition**

## CSI 4133 Computer Methods in Picture Processing and Analysis

Fall 2024

School of Electrical Engineering and Computer Science
University of Ottawa

Course Coordinator: Pengcheng Xi, Ph.D.


**Student Name:** Zijun Ye

**Student Number:** 300168065

**Submission Date:** November 21th, 2024


## **Methodology**


### **1. library used**


*   Python OpenCV
*   Deepface


Note
- The code able to detect total 7 type emotion: Face detect: Neutral, Suprise, Happy, Sad, disgusting, fear, angry
(p.s Disgust and fear a bit hard to detect:)


### **2. Steps**






**1. Import Modules**

```
import cv2
from deepface import DeepFace
import numpy as np
```

- In here we major use opencv, deepface and numerical library

**2. Set up Real time Webcam and Output video**

- initializing web cam and set the camera resolution to Full HD (1920x1080)

```
cam = cv2.VideoCapture(0)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

```

- In following code, i initialize the output frame width and height, as well as video format and frame rate:




```
frame_width = int(cam.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('result.mp4', fourcc, 10.0, (frame_width, frame_height))

```




**3. Set up Facial recogniztion**

- in here, i used the OpenCV's haarcasades library's Haar Cascade Model



```
facecascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
```



**4. Capture and Processing Frame By using while loop:**

```
while True:
    ret, frame = cam.read()

```

- Exit Condition: the loop will be exist by enter any key

```
if cv2.waitKey(1) != -1:
    print("Key pressed. Exiting loop...")
    break

```

- End of the program, the record video will be the result 'out' video

```
cam.release()
out.release()
cv2.destroyAllWindows()

```

**5. Emotion Analysis with Deepface**

- in here I used the deepface method 'analyze', which analyzes the current frame to predict emotion

- enforce_detection = false which means proceeds even if no face is detected


- And I specify select 'emotion' attribute to read from analyze

```
result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
emotion = result[0]['dominant_emotion']
```


**6. Improvement of emotion detection**

- In here, I improve the code by selecting most frequent detect emotion to be the final emotion result by using list store list of detect emotion and finding max frequence one to be the final emotion detected:

- first I created the list outside of while loop:

```
recent_predictions = []
```

- After deepface's analyze face, it append the emotion to list and final only select the most frequent detected one


```
recent_predictions.append(emotion)
if len(recent_predictions) > 5:
    recent_predictions.pop(0)
smoothed_emotion = max(set(recent_predictions), key=recent_predictions.count)
```

**7. Detect faces and draw rectangle around it**

- In the following code, I converts the frame to grayscale, which is required by Haar Cascade.

- Detects faces using detectMultiScale with scaling factor 1.1 and minimum neighbors 4.
For each detected face, draws a green rectangle ((0, 255, 0) in BGR format) around it.



```
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
faces = facecascade.detectMultiScale(gray, 1.1, 4)
for (x, y, w, h) in faces:
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

```



**8. Final step: display the smoothed emotion then repeat:**

Displays the smoothed emotion at the top-left corner of the frame in green text with a thickness of 3.



```
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(frame, smoothed_emotion, (0, 50), font, 2, (0, 255, 0), 3, cv2.LINE_4)

```



## **Source Code**

In [ ]:
# CSI41033 Project Part B
# Name: Zijun Ye 
# Student Number: 300168065

# Import the required modules 
import cv2 
from deepface import DeepFace
import numpy as np

# Open webcam + save the video after close the window 
# Open the default camera
cam = cv2.VideoCapture(0)
# Set camera resolution to higher quality (e.g., 1920x1080)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

# Haar Cascade File Path 
facecascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Get the default frame width and height
frame_width = int(cam.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cam.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('result.mp4', fourcc, 10.0, (frame_width, frame_height))

recent_predictions = []

try:
    while True:
        ret, frame = cam.read()

        if not ret:
            print("Failed to grab frame. Exiting...")
            break

        # Face Detection 
        result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
        # result = DeepFace.analyze(frame, actions=['emotion','race','gender','age'], enforce_detection=False)

        emotion = result[0]['dominant_emotion']

        recent_predictions.append(emotion)
        if len(recent_predictions) > 5:  # Keep the last 5 predictions
            recent_predictions.pop(0)
        smoothed_emotion = max(set(recent_predictions), key=recent_predictions.count)

        # Draw rectangle
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = facecascade.detectMultiScale(gray, 1.1, 4)
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Font settings
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 2
        font_thickness = 3
        text_color = (0, 255, 0)

        # Calculate text size
        text_size = cv2.getTextSize(smoothed_emotion, font, font_scale, font_thickness)[0]
        text_x = (frame_width - text_size[0]) // 2  # Center horizontally
        text_y = 50  # Position near the top

        # Write the emotion text at the top center
        cv2.putText(frame, smoothed_emotion, (text_x, text_y), font, font_scale, text_color, font_thickness, cv2.LINE_4)

        # Write the frame to the output file
        out.write(frame)

        # Display the captured frame
        cv2.imshow('Face Detection and Analysis', frame)

        # Check for the Esc key (27) to break the loop
        if cv2.waitKey(1) & 0xFF == 27:
            print("Esc key pressed. Exiting loop...")
            break
finally:
    # Release the capture and writer objects
    print("Releasing resources...")
    cam.release()
    out.release()
    cv2.destroyAllWindows()

## **Reference**



*  **Greeksforgeeks** for live video capture: https://www.geeksforgeeks.org/python-opencv-capture-video-from-camera/

* **ChatGPT-4o** for improvement of detection accuracy by using Filter Noisy Predictions methods:
  - Prompt: does there have any way to increase the accuracy of the emotion detection in deepface?

* **Medium article** for usage of deepface: https://medium.com/@adarshkumarchaurasiya/recognition-model-facial-emotion-recogination-using-deepface-analyzer-c305465be123

* **Greeksforgeeks** for usage deepface: https://www.geeksforgeeks.org/facial-expression-detection-using-deepface-module-in-python/

* **Github** for initial research of deepface: https://github.com/serengil/deepface

